In [1]:
#this notebook is there for the plotting the Sankey diagram based on the aggregated MFA dataframe generated in SUT-to-MFA_main-pipeline.ipynb (and TC-gamma-push_pipeline.ipynb)

In [2]:
#general operations
import pandas as pd
import matplotlib
import plotly.graph_objects as go
import matplotlib.colors as mcolors

# Sankey diagram

In [3]:
#extract unique elements in From of df_MFA_scaled
df_MFA_sorted = pd.read_csv('Results/df_MFA_coloured&scaled.csv',index_col=0)
df_MFA_sorted

,Product,From,To,Quantity (t),Colour_HEX
0,Buildings and infrastructure,Utilities & Construction,Capital formation,1.003315e+06,#ad494a
1,Buildings and infrastructure,Utilities & Construction,Final consumption expenditure,3.730346e+04,#ad494a
2,Chemicals and synthetic products without Plast...,Agriculture,Capital formation,1.361821e+03,#e7969c
3,Chemicals and synthetic products without Plast...,Agriculture,Environment,2.585535e+01,#e7969c
4,Chemicals and synthetic products without Plast...,International imports,Agriculture,6.051968e+02,#e7969c
...,...,...,...,...,...
1042,Wood-related products,Services,Capital formation,5.121609e+01,#393b79
1043,Wood-related products,Utilities & Construction,Capital formation,6.958755e+03,#393b79
1044,Wood-related products,Utilities & Construction,Environment,1.154207e+02,#393b79
1045,Wood-related products,Utilities & Construction,Waste,8.169333e+02,#393b79


In [4]:
#show all unique products in df_MFA_sorted to define colours for flows
unique_prod = df_MFA_sorted['Colour_HEX'].unique()
print(unique_prod)
#reduce data frame to only 'Quantity (t)', 'From', 'To', and 'Colour_HEX' that are needed for the Sankey
links = pd.DataFrame(df_MFA_sorted, columns=['Quantity (t)', 'From', 'To', 'Colour_HEX'])

#remove rows where 'From' (and 'To') = "NaN"
links = links.dropna(subset=['From'])

links

['#ad494a' '#e7969c' '#7b4173' '#a55194' '#bd9e39' '#d6616b' '#9c9ede'
 '#ce6dbd' '#5254a3' '#8c6d31' '#637939' '#e7ba52' '#8ca252' '#e7cb94'
 '#cedb9c' '#d3d3d3' '#b5cf6b' '#843c39' '#393b79']


,Quantity (t),From,To,Colour_HEX
0,1.003315e+06,Utilities & Construction,Capital formation,#ad494a
1,3.730346e+04,Utilities & Construction,Final consumption expenditure,#ad494a
2,1.361821e+03,Agriculture,Capital formation,#e7969c
3,2.585535e+01,Agriculture,Environment,#e7969c
4,6.051968e+02,International imports,Agriculture,#e7969c
...,...,...,...,...
1042,5.121609e+01,Services,Capital formation,#393b79
1043,6.958755e+03,Utilities & Construction,Capital formation,#393b79
1044,1.154207e+02,Utilities & Construction,Environment,#393b79
1045,8.169333e+02,Utilities & Construction,Waste,#393b79


In [5]:
#get categorical colour map for 25 distinct colours
cmap1 = matplotlib.colormaps.get_cmap('tab20')
cmap2 = matplotlib.colormaps.get_cmap('tab20b')  #because we need more than 20 distinct colours...
#use colour map related to product archetypes
color_map = {

    #Greens — chemicals, plastics
    'Plastic resins': mcolors.to_hex('lightgrey'),  #or 'grey'
    'Plastic bags': mcolors.to_hex(cmap2(4 / 19)),
    'Plastic films and non-rigid sheets': mcolors.to_hex(cmap2(5 / 19)),
    'Motor vehicle plastic parts': mcolors.to_hex(cmap2(3 / 19)),
    # Remaining plastic products — reuse greens
    'Plastic and foam building and construction materials': mcolors.to_hex(cmap2(8 / 19)),
    'Foam products (except for construction)':             mcolors.to_hex(cmap2(9 / 19)),
    'Plastic bottles':                                     mcolors.to_hex(cmap2(10 / 19)),
    'Plastic products, n.e.c.':                            mcolors.to_hex(cmap2(11 / 19)),
    'Rubber and plastic hoses and belts':                  mcolors.to_hex(cmap2(6 / 19)),
    'Plastic profile shapes':                              mcolors.to_hex(cmap2(7 / 19)),
    'Waste and scrap of plastic and rubber':               mcolors.to_hex(cmap2(12 / 19)),
    #Blues - wood, paper, natural materials
    'Wood-related products': mcolors.to_hex(cmap2(0 / 19)),
    'Paper-related products': mcolors.to_hex(cmap2(1 / 19)),
    #'Processed food products': mcolors.to_hex(cmap2(2 / 19)),
    #Other — environment, waste
    #'Environment': mcolors.to_hex(cmap2(8 / 19)),
    #'Waste': mcolors.to_hex(cmap2(9 / 19)),
    #Reds — infrastructure, machinery
    #'Capital formation': mcolors.to_hex(cmap2(15 / 19)),
    'Buildings and infrastructure': mcolors.to_hex(cmap2(13 / 19)),
    'Hardware, machinery, and equipment': mcolors.to_hex(cmap2(14 / 19)),
    'Chemicals and synthetic products without Plastic resins': mcolors.to_hex(cmap2(15 / 19)),
    #Pinks/reds — consumer goods, clothes
    'Clothes and textiles': mcolors.to_hex(cmap2(16 / 19)),
    'Consumer goods': mcolors.to_hex(cmap2(17 / 19)),
    'Other supplies': mcolors.to_hex(cmap2(18 / 19)),

}

In [6]:
#create unique list of source/'From' and target/'To' for plotly
unique_from_to = list(pd.unique(links[['From', 'To']].values.ravel('K')))

#create a list of unique nodes
unique_nodes = pd.concat([links['From'], links['To']]).unique()
node_index = {node: i for i, node in enumerate(unique_nodes)}

#create new columns for indices in a safe way
links = links.assign(
    FromIndex=links['From'].map(node_index),
    ToIndex=links['To'].map(node_index))

#plot plotly Sankey
fig = go.Figure(data=[go.Sankey(
    valueformat=None,
    valuesuffix='t',  #all flows in unit tonnes
    node=dict(
        pad=15,
        thickness=10,
        line=dict(color="black", width=0.5),
        label=unique_from_to,
        color='#808080'
    ),
    link=dict(
        source=links['FromIndex'],  # indices correspond to labels
        target=links['ToIndex'],
        value=links['Quantity (t)'],
        color=links['Colour_HEX']
    ))])
#to define Node Position manually: https://plotly.com/python/sankey-diagram/


#create legend
for label, colour in color_map.items():  #need to append 'Other product, grey'
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=10, color=colour),
        name=label,
        showlegend=True
    ))

fig.update_layout(
    #title_text="Sankey diagram of plastic flows through the Canadian economy in 2019",
    font_size=10,
    font_color="black",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    showlegend=True,
    legend=dict(
        title='Legend',
        y=1,
        x=1.05),
    width=2000,
    height=1000,
    font=dict(size=14),
    plot_bgcolor='rgba(0,0,0,0)',  # Transparent plot area
    paper_bgcolor='rgba(0, 0, 0, 0)',  # Transparent figure background
)


fig.show()

In [7]:
#save figure in html format in files
fig.write_html(f'SUT-to-MFA_without-markets.html')